# Assignment 3: Sanskrit-English Sentence Embeddings

This notebook generates sentence-level cross-lingual embeddings for parallel Sanskrit-English text, optimizes embedding dimensionality, evaluates semantic alignment using cosine similarity, and visualizes 100 aligned pairs with t-SNE.

## 1) Install Dependencies

Run this cell once in a fresh environment.

In [6]:
# If running for the first time, install required packages.
# Keep external dependencies minimal as required by the assignment.
%pip install -q torch pandas numpy scikit-learn matplotlib sentence-transformers

Note: you may need to restart the kernel to use updated packages.


## 2) Imports and Configuration

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Pre-trained multilingual sentence embedding model (no external API calls).
MODEL_NAME = "sentence-transformers/LaBSE"

# Candidate dimensions for balancing compactness vs semantic quality.
CANDIDATE_DIMS = [64, 128, 256, 384, 512, 768]

print(f"PyTorch device: {DEVICE}")

PyTorch device: cpu


## 3) Load and Align Parallel Data

Expected folder:
- `data-a3` under the current working directory

Expected files inside `data-a3`:
- train_sa.csv, dev_sa.csv, test_sa.csv
- train_en.csv, dev_en.csv, test_en.csv

The Sanskrit and English rows are aligned using Source_id.

In [8]:
def find_dataset_root() -> Path:
    data_root = Path.cwd() / "data-a3"
    required = {
        "train_sa_10000.csv", "dev_sa_1000.csv", "test_sa_1000.csv",
        "train_en_10000.csv", "dev_en_1000.csv", "test_en_1000.csv",
    }

    if not data_root.exists():
        raise FileNotFoundError(
            f"Expected dataset folder not found: {data_root}. "
            "Please create/populate 'data-a3' under the current directory."
        )

    names = {p.name for p in data_root.glob("*.csv")}
    missing = sorted(required - names)
    if missing:
        raise FileNotFoundError(
            f"Missing required files in {data_root}: {', '.join(missing)}"
        )

    return data_root


def load_split(split: str, data_root: Path) -> pd.DataFrame:
    sa_path = data_root / f"{split}_sa.csv"
    en_path = data_root / f"{split}_en.csv"

    sa_df = pd.read_csv(sa_path)
    en_df = pd.read_csv(en_path)

    # Keep only required columns and normalize text to string.
    sa_df = sa_df[["Source_id", "Sentence_sa"]].copy()
    en_df = en_df[["Source_id", "Sentence_en"]].copy()
    sa_df["Sentence_sa"] = sa_df["Sentence_sa"].astype(str)
    en_df["Sentence_en"] = en_df["Sentence_en"].astype(str)

    merged = sa_df.merge(en_df, on="Source_id", how="inner")
    merged = merged.dropna(subset=["Sentence_sa", "Sentence_en"]).reset_index(drop=True)
    return merged


data_root = find_dataset_root()
print(f"Dataset root: {data_root}")

train_df = load_split("train", data_root)
dev_df = load_split("dev", data_root)
test_df = load_split("test", data_root)

print(f"Train pairs: {len(train_df)}")
print(f"Dev pairs:   {len(dev_df)}")
print(f"Test pairs:  {len(test_df)}")

display(train_df.head(3))

Dataset root: c:\Rikin\Study\IIT-Futurence\2025-26\trimester-2\Deep Learning\code\dl-learning\data-a3


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Rikin\\Study\\IIT-Futurence\\2025-26\\trimester-2\\Deep Learning\\code\\dl-learning\\data-a3\\train_sa.csv'

## 4) Generate Base Sentence Embeddings

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f"Loaded model: {MODEL_NAME}")


def encode_sentences(sentences):
    emb = model.encode(
        sentences,
        batch_size=32,
        show_progress_bar=True,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    return emb.detach().to(torch.float32)

sa_train_base = encode_sentences(train_df["Sentence_sa"].tolist())
en_train_base = encode_sentences(train_df["Sentence_en"].tolist())
sa_dev_base = encode_sentences(dev_df["Sentence_sa"].tolist())
en_dev_base = encode_sentences(dev_df["Sentence_en"].tolist())
sa_test_base = encode_sentences(test_df["Sentence_sa"].tolist())
en_test_base = encode_sentences(test_df["Sentence_en"].tolist())

print(f"Base embedding dimension: {sa_train_base.shape[1]}")

## 5) Select Embedding Dimension on Dev Set

We evaluate multiple dimensions and choose the smallest dimension whose dev cosine score is within 99% of the best dev score.

In [ ]:
def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / torch.clamp(torch.linalg.norm(x, dim=1, keepdim=True), min=eps)


def avg_pair_cosine(x: torch.Tensor, y: torch.Tensor) -> float:
    x_n = l2_normalize(x)
    y_n = l2_normalize(y)
    return float((x_n * y_n).sum(dim=1).mean().item())


def fit_torch_pca(x: torch.Tensor, n_components: int):
    # Center train embeddings and learn principal directions with torch.pca_lowrank.
    mean = x.mean(dim=0, keepdim=True)
    x_centered = x - mean
    _, _, v = torch.pca_lowrank(x_centered, q=n_components, center=False)
    components = v[:, :n_components]
    return {"mean": mean, "components": components}


def apply_torch_pca(x: torch.Tensor, pca_state):
    x_centered = x - pca_state["mean"]
    return x_centered @ pca_state["components"]


base_dim = sa_train_base.shape[1]
valid_dims = [d for d in CANDIDATE_DIMS if d <= base_dim]
if base_dim not in valid_dims:
    valid_dims.append(base_dim)
valid_dims = sorted(set(valid_dims))

# Fit PCA only on train embeddings to avoid dev/test leakage during model selection.
joint_train = torch.cat([sa_train_base, en_train_base], dim=0)

records = []
pca_by_dim = {}

for d in valid_dims:
    if d == base_dim:
        sa_dev_proj = sa_dev_base
        en_dev_proj = en_dev_base
        pca_state = None
    else:
        pca_state = fit_torch_pca(joint_train, d)
        sa_dev_proj = apply_torch_pca(sa_dev_base, pca_state)
        en_dev_proj = apply_torch_pca(en_dev_base, pca_state)

    score = avg_pair_cosine(sa_dev_proj, en_dev_proj)
    records.append({"dimension": d, "dev_avg_cosine": score})
    pca_by_dim[d] = pca_state

results_df = pd.DataFrame(records).sort_values("dimension").reset_index(drop=True)
display(results_df)

best_score = results_df["dev_avg_cosine"].max()
threshold = 0.99 * best_score
chosen_dim = int(results_df.loc[results_df["dev_avg_cosine"] >= threshold, "dimension"].min())

print(f"Best dev cosine score: {best_score:.6f}")
print(f"Selection threshold (99% of best): {threshold:.6f}")
print(f"Chosen embedding dimension: {chosen_dim}")

## 6) Final Embeddings and Test Cosine Similarity

After choosing the dimension on dev data, we evaluate the final average cosine similarity on test pairs.

In [ ]:
if chosen_dim == base_dim:
    sa_test_final = sa_test_base
    en_test_final = en_test_base
else:
    pca_final = pca_by_dim[chosen_dim]
    sa_test_final = apply_torch_pca(sa_test_base, pca_final)
    en_test_final = apply_torch_pca(en_test_base, pca_final)

final_avg_cosine = avg_pair_cosine(sa_test_final, en_test_final)

print(f"Embedding dimension used: {chosen_dim}")
print(f"Final average cosine similarity (test): {final_avg_cosine:.6f}")

## 7) t-SNE Visualization for 100 Sample Pairs

This plot shows 2D projections of Sanskrit and English embeddings for 100 aligned sentence pairs.

In [ ]:
sample_n = min(100, len(test_df))
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(test_df), size=sample_n, replace=False)
sample_idx_t = torch.as_tensor(sample_idx, dtype=torch.long, device=sa_test_final.device)

sa_sample = sa_test_final.index_select(0, sample_idx_t).detach().cpu().numpy()
en_sample = en_test_final.index_select(0, sample_idx_t).detach().cpu().numpy()

both = np.vstack([sa_sample, en_sample])

# Perplexity must be < number of samples.
perplexity = min(30, max(5, (2 * sample_n) // 10))
if perplexity >= len(both):
    perplexity = max(2, len(both) - 1)

tsne = TSNE(
    n_components=2,
    random_state=SEED,
    init="pca",
    learning_rate="auto",
    perplexity=perplexity,
)
coords = tsne.fit_transform(both)

sa_xy = coords[:sample_n]
en_xy = coords[sample_n:]

plt.figure(figsize=(11, 8))
plt.scatter(sa_xy[:, 0], sa_xy[:, 1], alpha=0.75, label="Sanskrit", marker="o")
plt.scatter(en_xy[:, 0], en_xy[:, 1], alpha=0.75, label="English", marker="^")

# Draw faint lines to show aligned pairs in 2D projection.
for i in range(sample_n):
    plt.plot(
        [sa_xy[i, 0], en_xy[i, 0]],
        [sa_xy[i, 1], en_xy[i, 1]],
        linewidth=0.6,
        alpha=0.25,
        color="gray",
    )

plt.title(f"t-SNE of {sample_n} Sanskrit-English Aligned Pairs")
plt.xlabel("t-SNE dimension 1")
plt.ylabel("t-SNE dimension 2")
plt.legend()
plt.grid(alpha=0.2)
plt.show()